# Bài 2 — Group-Factorized Centered CLIPConceptHead + Differential Top-K Concept Arbitration

Khắc phục triệt để hạn chế của các bản thử trước để tận dụng trực tiếp ưu thế **Unseen Concept AUC ~ 0.83–0.97** của `CLIPConceptHead`:
1. **Centered & Normalized CLIP Primitive Space**: Trừ vector trung bình $\bar{E}$ của 82 primitive trước khi L2-normalize để phá bỏ *anisotropic cone* của CLIP text encoder.
2. **Group-Factorized CLIPConceptHead ($G=7$ nhóm bộ phận cơ thể)**: Tách phép chiếu thành $W_{\text{global}} z_c + W_{g(p)} z_c$ cho 7 nhóm (`hand: 35, interaction: 15, body: 12, arm: 6, torso: 5, leg: 5, head: 4`).
3. **Cross-Modal Soft LLM Distillation**: Bổ sung giám sát mềm từ tương đồng giữa văn bản mô tả lớp của LLM và văn bản primitive cho các primitive chưa xuất hiện ở tập seen.
4. **IDF-Weighted Differential Top-$K$ Arbitration**: Giữ nguyên 100% chuỗi RNG và quỹ đạo DiT của Flora, chỉ phân xử trong Top-$K$ ($K=3$) ứng viên của DiT bằng các **khái niệm phân biệt độc quyền** ($\mathcal{R}_{c_k} \setminus \mathcal{R}_{c_1}$ so với $\mathcal{R}_{c_1} \setminus \mathcal{R}_{c_k}$), tự động triệt tiêu các khái niệm trùng nhau.

**Trước khi chạy**: commit và push `zeroshot/concept_map/` lên GitHub.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess, time, re
import numpy as np, pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/BaoHuy/zeroshot'
FEAT_ARCHIVE = '/content/drive/MyDrive/BaoHuy/flora_pretrain.zip'
OUT_DIR = f'{DRIVE_DIR}/concept_map_step1'
EXTRACT = '/content/flora_features'
os.makedirs(OUT_DIR, exist_ok=True)

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], check=True, stdout=open(f'{OUT_DIR}/thesis_commit.txt', 'w'))

FLORA = f'{REPO}/zeroshot/Flora'
CMAP = f'{REPO}/zeroshot/concept_map'
need_flora = ['nesy_common.py', 'kg_semantics.py', 'kg/primitive_v2.json', 'kg/kg_rules_v2.json', 'semantics/ntu60_llm.txt', 'semantics/ntu120_llm.txt']
need_cmap = ['model_cr.py', 'trainer_cr.py', 'run_cr.py', 'model_cfm.py', 'trainer_cfm.py', 'run_cfm.py']
missing = [f'Flora/{p}' for p in need_flora if not os.path.exists(f'{FLORA}/{p}')]
missing += [f'concept_map/{p}' for p in need_cmap if not os.path.exists(f'{CMAP}/{p}')]
assert not missing, f'Repo chưa có {missing}: commit và push zeroshot/concept_map trước'
print('repo OK')

In [ ]:
!pip -q install transformers pyyaml pandas scikit-learn

stub = f'{FLORA}/models/shiftgcn/Temporal_shift/cuda/shift_cuda.py'
if not os.path.exists(stub):
    with open(stub, 'w') as f:
        f.write('# stub: Shift-GCN CUDA op is never called when use_features=True\n')

!python -c "import sys; sys.path[:0]=['{FLORA}','{CMAP}']; from model_cr import ConceptArbitratedMappingNet, GroupFactorizedCLIPConceptHead; from trainer_cr import ConceptArbitratedTrainer; print('import GF-CLIPConceptHead + Differential Top-K Arbitration OK')"

## Giải nén feature 256-D đã trích sẵn

In [ ]:
assert os.path.exists(FEAT_ARCHIVE), f'Không thấy {FEAT_ARCHIVE} trên Drive'
!ls -lh {FEAT_ARCHIVE}

FEAT_NAMES = ('star_features', 'synse_features')

def find_feat(name):
    hits = [h for h in glob.glob(f'{EXTRACT}/**/{name}', recursive=True) if '__MACOSX' not in h]
    return hits[0] if hits else None

if not any(find_feat(n) for n in FEAT_NAMES):
    os.makedirs(EXTRACT, exist_ok=True)
    t0 = time.time()
    subprocess.run(['unzip', '-q', '-o', FEAT_ARCHIVE, '-x', '__MACOSX/*', '*/._*', '._*', '-d', EXTRACT], check=True)
    print(f'giải nén xong trong {time.time() - t0:.0f}s')

assert any(find_feat(n) for n in FEAT_NAMES), f'Trong {FEAT_ARCHIVE} không có star_features hay synse_features'
os.makedirs(f'{FLORA}/skeleton_features', exist_ok=True)
for name in FEAT_NAMES:
    src_dir = find_feat(name)
    link = f'{FLORA}/skeleton_features/{name}'
    if src_dir is None:
        continue
    if os.path.islink(link):
        os.remove(link)
    os.symlink(src_dir, link)
    print(name, sorted(f for f in os.listdir(link) if not f.startswith('._')))

## Thực nghiệm Group-Factorized Centered CLIPConceptHead + Differential Top-K Arbitration (STAR features, 48/12 & 110/10, seed 1)

Chạy 4 biến thể ablation (trong mỗi lần chạy, cột `a=0.00` của `AlphaSweep` chính là **Flora baseline thuần** trong cùng chuỗi RNG để đối chiếu trực tiếp với `a in {0.03, 0.05, 0.08, 0.12, 0.20}`):
- `gf_arb`: Mô hình đầy đủ (`center_clip=1, use_groups=1, lambda_soft=0.2, arb_topk=3, arb_alpha=0.08`)
- `gf_no_soft`: Tắt Soft LLM Text Distillation (`lambda_soft=0.0`)
- `gf_no_group`: Tắt 7 đầu chiếu bộ phận cơ thể (`use_groups=0`)
- `gf_uncentered`: Tắt Centering không gian CLIP (`center_clip=0, use_groups=0, lambda_soft=0.0`)

In [ ]:
!python {CMAP}/run_cr.py --feat star --tasks 48 110 --seeds 1 --variants gf_arb gf_no_soft gf_no_group gf_uncentered --out-root {OUT_DIR} --force

## Xem chi tiết Concept Inference (`[GF-EVAL]`) & Demo Top-5 concept (`[GF-DEMO]`)

In [ ]:
for variant in ('gf_arb', 'gf_no_soft', 'gf_no_group', 'gf_uncentered'):
    for split_task in ('ntu60_xsub_seen48_unseen12', 'ntu120_xsub_seen110_unseen10'):
        log_path = f'{OUT_DIR}/star/{variant}/{split_task}_ZSL_seed1/run.log'
        if os.path.exists(log_path):
            print(f'=== {variant} | {split_task} ===')
            for line in open(log_path, encoding='utf-8'):
                if '[GF-EVAL]' in line or '[GF-DEMO]' in line or 'Best ZSL' in line:
                    print(line.strip())